# Artificial Neural Networks with topologic_fast

This notebook demonstrates how to use artificial neural networks (ANNs) for classification and regression tasks with topological feature data generated using `topologic_fast`.

## Overview

**Note:** topologicpy provides a built-in `ANN` class that wraps PyTorch. In topologic_fast, we use PyTorch directly, demonstrating how to:
- Extract features from topology using topologic_fast
- Build and train neural networks with PyTorch
- Evaluate model performance
- Visualize learning curves and predictions

This approach provides more flexibility and integrates with the broader PyTorch ecosystem.

In [ ]:
# Import required libraries
import topologic_fast as tf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Check for PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f"PyTorch is available (version {torch.__version__})")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed. Install with: pip install torch")
    print("This notebook will demonstrate concepts without running training.")

# Check for sklearn (for train/test split and metrics)
try:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn not installed. Install with: pip install scikit-learn")

## 1. Generate Training Data from Topology

We'll create various 3D shapes using topologic_fast and extract features for classification.

In [ ]:
def extract_cell_features(cell):
    """Extract features from a Cell for ML."""
    volume = cell.Volume()
    area = cell.Area()
    compactness = cell.Compactness()
    
    # Number of faces, edges, vertices
    n_faces = len(cell.Faces())
    n_edges = len(cell.Edges())
    n_vertices = len(cell.Vertices())
    
    # Center of mass
    com = cell.CenterOfMass()
    
    # Surface to volume ratio
    sv_ratio = area / volume if volume > 0 else 0
    
    return {
        'volume': volume,
        'area': area,
        'compactness': compactness,
        'n_faces': n_faces,
        'n_edges': n_edges,
        'n_vertices': n_vertices,
        'com_x': com[0],
        'com_y': com[1],
        'com_z': com[2],
        'sv_ratio': sv_ratio
    }

# Test feature extraction
test_box = tf.Cell.Box(0, 0, 0, 2, 3, 4)
features = extract_cell_features(test_box)
print("Features extracted from a 2x3x4 box:")
for k, v in features.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
def generate_shape_dataset(n_samples_per_class=50):
    """Generate a dataset of different 3D shapes with labels."""
    data = []
    labels = []
    
    np.random.seed(42)
    
    # Class 0: Cubes (equal dimensions)
    for _ in range(n_samples_per_class):
        size = np.random.uniform(1, 5)
        # Add small random variations
        w = size * np.random.uniform(0.95, 1.05)
        l = size * np.random.uniform(0.95, 1.05)
        h = size * np.random.uniform(0.95, 1.05)
        cell = tf.Cell.Box(0, 0, 0, w, l, h)
        features = extract_cell_features(cell)
        data.append(features)
        labels.append(0)  # Cube
    
    # Class 1: Flat boxes (height much smaller)
    for _ in range(n_samples_per_class):
        w = np.random.uniform(2, 6)
        l = np.random.uniform(2, 6)
        h = np.random.uniform(0.3, 0.8)  # Flat
        cell = tf.Cell.Box(0, 0, 0, w, l, h)
        features = extract_cell_features(cell)
        data.append(features)
        labels.append(1)  # Flat
    
    # Class 2: Tall boxes (height much larger)
    for _ in range(n_samples_per_class):
        w = np.random.uniform(1, 2)
        l = np.random.uniform(1, 2)
        h = np.random.uniform(5, 10)  # Tall
        cell = tf.Cell.Box(0, 0, 0, w, l, h)
        features = extract_cell_features(cell)
        data.append(features)
        labels.append(2)  # Tall
    
    # Class 3: Long boxes (one horizontal dimension much larger)
    for _ in range(n_samples_per_class):
        w = np.random.uniform(5, 10)  # Long
        l = np.random.uniform(1, 2)
        h = np.random.uniform(1, 2)
        cell = tf.Cell.Box(0, 0, 0, w, l, h)
        features = extract_cell_features(cell)
        data.append(features)
        labels.append(3)  # Long
    
    return pd.DataFrame(data), np.array(labels)

# Generate dataset
df, labels = generate_shape_dataset(n_samples_per_class=100)
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
class_names = ['Cube', 'Flat', 'Tall', 'Long']
for i, name in enumerate(class_names):
    print(f"  {name}: {np.sum(labels == i)}")

df.head()

In [ ]:
# Visualize feature distributions
fig = make_subplots(rows=2, cols=2, 
                    subplot_titles=['Volume Distribution', 'Compactness Distribution',
                                   'Surface/Volume Ratio', 'Number of Faces'])

colors = ['red', 'green', 'blue', 'orange']

for i, (name, color) in enumerate(zip(class_names, colors)):
    mask = labels == i
    
    fig.add_trace(go.Histogram(x=df.loc[mask, 'volume'], name=name, 
                               marker_color=color, opacity=0.6), row=1, col=1)
    fig.add_trace(go.Histogram(x=df.loc[mask, 'compactness'], name=name,
                               marker_color=color, opacity=0.6, showlegend=False), row=1, col=2)
    fig.add_trace(go.Histogram(x=df.loc[mask, 'sv_ratio'], name=name,
                               marker_color=color, opacity=0.6, showlegend=False), row=2, col=1)
    fig.add_trace(go.Histogram(x=df.loc[mask, 'n_faces'], name=name,
                               marker_color=color, opacity=0.6, showlegend=False), row=2, col=2)

fig.update_layout(
    title="Feature Distributions by Shape Class",
    height=600,
    width=900,
    barmode='overlay'
)

fig.show()

## 2. Prepare Data for Training

In [ ]:
if SKLEARN_AVAILABLE and TORCH_AVAILABLE:
    # Select features for training
    feature_cols = ['volume', 'area', 'compactness', 'n_faces', 'n_edges', 'n_vertices', 'sv_ratio']
    X = df[feature_cols].values
    y = labels
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"Training set: {X_train_scaled.shape[0]} samples")
    print(f"Validation set: {X_val_scaled.shape[0]} samples")
    print(f"Test set: {X_test_scaled.shape[0]} samples")
    print(f"\nFeatures: {feature_cols}")
else:
    print("Skipping data preparation (missing dependencies)")

In [ ]:
if TORCH_AVAILABLE:
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled)
    y_train_tensor = torch.LongTensor(y_train)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.LongTensor(y_val)
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    y_test_tensor = torch.LongTensor(y_test)
    
    # Create data loaders
    batch_size = 16
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    print(f"Created data loaders with batch size {batch_size}")

## 3. Define Neural Network Architecture

We'll create a simple feedforward neural network with configurable hidden layers.

In [ ]:
if TORCH_AVAILABLE:
    class ShapeClassifier(nn.Module):
        """A simple feedforward neural network for shape classification."""
        
        def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.2, batch_norm=True):
            super(ShapeClassifier, self).__init__()
            
            layers = []
            prev_dim = input_dim
            
            for hidden_dim in hidden_dims:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                if batch_norm:
                    layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                prev_dim = hidden_dim
            
            layers.append(nn.Linear(prev_dim, num_classes))
            
            self.network = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.network(x)
    
    # Create model
    input_dim = len(feature_cols)
    hidden_dims = [64, 64]
    num_classes = len(class_names)
    
    model = ShapeClassifier(input_dim, hidden_dims, num_classes, dropout=0.2)
    print(model)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params}")
    print(f"Trainable parameters: {trainable_params}")

## 4. Train the Model

In [ ]:
if TORCH_AVAILABLE:
    def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3, 
                    weight_decay=1e-4, early_stopping_patience=15):
        """Train the model with early stopping."""
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        
        history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': []
        }
        
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        
        for epoch in range(epochs):
            # Training
            model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * X_batch.size(0)
                _, predicted = torch.max(outputs, 1)
                train_correct += (predicted == y_batch).sum().item()
                train_total += y_batch.size(0)
            
            train_loss /= train_total
            train_acc = train_correct / train_total
            
            # Validation
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    
                    val_loss += loss.item() * X_batch.size(0)
                    _, predicted = torch.max(outputs, 1)
                    val_correct += (predicted == y_batch).sum().item()
                    val_total += y_batch.size(0)
            
            val_loss /= val_total
            val_acc = val_correct / val_total
            
            # Update scheduler
            scheduler.step(val_loss)
            
            # Save history
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs}: Train Loss={train_loss:.4f}, "
                      f"Train Acc={train_acc:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
            
            if patience_counter >= early_stopping_patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        
        # Load best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        return history
    
    # Train the model
    print("Training the model...\n")
    history = train_model(model, train_loader, val_loader, epochs=150, lr=1e-3)
    print(f"\nTraining complete. Best validation accuracy: {max(history['val_acc']):.4f}")

## 5. Visualize Training History

In [ ]:
if TORCH_AVAILABLE:
    fig = make_subplots(rows=1, cols=2, subplot_titles=['Loss', 'Accuracy'])
    
    # Loss plot
    fig.add_trace(go.Scatter(
        y=history['train_loss'],
        mode='lines',
        name='Train Loss',
        line=dict(color='blue')
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        y=history['val_loss'],
        mode='lines',
        name='Val Loss',
        line=dict(color='orange')
    ), row=1, col=1)
    
    # Accuracy plot
    fig.add_trace(go.Scatter(
        y=history['train_acc'],
        mode='lines',
        name='Train Acc',
        line=dict(color='blue'),
        showlegend=False
    ), row=1, col=2)
    
    fig.add_trace(go.Scatter(
        y=history['val_acc'],
        mode='lines',
        name='Val Acc',
        line=dict(color='orange'),
        showlegend=False
    ), row=1, col=2)
    
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=1, col=2)
    fig.update_yaxes(title_text="Loss", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", row=1, col=2)
    
    fig.update_layout(
        title="ANN Learning Curves",
        width=900,
        height=400
    )
    
    fig.show()

## 6. Evaluate on Test Set

In [ ]:
if TORCH_AVAILABLE and SKLEARN_AVAILABLE:
    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, y_pred = torch.max(outputs, 1)
        y_pred = y_pred.numpy()
    
    test_acc = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
if TORCH_AVAILABLE and SKLEARN_AVAILABLE:
    # Plot confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=class_names,
        y=class_names,
        colorscale='Blues',
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 16},
        showscale=True
    ))
    
    fig.update_layout(
        title="Confusion Matrix (Test Set)",
        xaxis_title="Predicted",
        yaxis_title="Actual",
        width=500,
        height=500
    )
    
    fig.show()

## 7. Predict on New Data

Let's create some new shapes and predict their classes.

In [ ]:
if TORCH_AVAILABLE:
    # Create new shapes for prediction
    new_shapes = [
        tf.Cell.Box(0, 0, 0, 3, 3, 3),      # Should be Cube
        tf.Cell.Box(0, 0, 0, 5, 5, 0.5),    # Should be Flat
        tf.Cell.Box(0, 0, 0, 1, 1, 8),      # Should be Tall
        tf.Cell.Box(0, 0, 0, 8, 1.5, 1.5),  # Should be Long
        tf.Cell.Box(0, 0, 0, 2.5, 2.5, 2),  # Ambiguous (near-cube)
    ]
    
    # Extract features
    new_features = []
    for shape in new_shapes:
        features = extract_cell_features(shape)
        new_features.append([features[col] for col in feature_cols])
    
    new_features = np.array(new_features)
    
    # Scale features
    new_features_scaled = scaler.transform(new_features)
    
    # Predict
    model.eval()
    with torch.no_grad():
        new_tensor = torch.FloatTensor(new_features_scaled)
        outputs = model(new_tensor)
        probs = torch.softmax(outputs, dim=1)
        _, predictions = torch.max(outputs, 1)
    
    # Display results
    shape_descriptions = [
        "3x3x3 (Cube)",
        "5x5x0.5 (Flat)",
        "1x1x8 (Tall)",
        "8x1.5x1.5 (Long)",
        "2.5x2.5x2 (Ambiguous)"
    ]
    
    print("Predictions for new shapes:\n")
    for i, desc in enumerate(shape_descriptions):
        pred_class = class_names[predictions[i]]
        prob = probs[i].numpy()
        print(f"{desc}:")
        print(f"  Predicted: {pred_class}")
        print(f"  Probabilities: {dict(zip(class_names, [f'{p:.3f}' for p in prob]))}")
        print()

## 8. Save and Load Model

In [ ]:
if TORCH_AVAILABLE:
    # Save model
    import tempfile
    import os
    
    model_path = os.path.join(tempfile.gettempdir(), 'shape_classifier.pt')
    
    # Save model state and metadata
    save_dict = {
        'model_state_dict': model.state_dict(),
        'input_dim': input_dim,
        'hidden_dims': hidden_dims,
        'num_classes': num_classes,
        'class_names': class_names,
        'feature_cols': feature_cols,
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist()
    }
    
    torch.save(save_dict, model_path)
    print(f"Model saved to: {model_path}")
    
    # Demonstrate loading
    loaded = torch.load(model_path)
    loaded_model = ShapeClassifier(
        loaded['input_dim'],
        loaded['hidden_dims'],
        loaded['num_classes']
    )
    loaded_model.load_state_dict(loaded['model_state_dict'])
    loaded_model.eval()
    
    print("Model loaded successfully!")

## 9. Summary

In this notebook, we demonstrated:

1. **Feature extraction from topology**: Using topologic_fast to extract geometric features from Cells

2. **Dataset generation**: Creating synthetic shape data with topologic_fast

3. **Neural network training**: Building and training a PyTorch classifier

4. **Model evaluation**: Using confusion matrices and classification reports

5. **Inference**: Making predictions on new shapes

### API Comparison

| topologicpy ANN | This Approach | Notes |
|-----------------|---------------|-------|
| `ANN.ByCSVPath()` | Load CSV with pandas | More flexible |
| `ANN.SetHyperparameters()` | Pass to train function | More Pythonic |
| `ANN.Train()` | Custom training loop | Full control |
| `ANN.Validate()` | Validation in loop | Integrated |
| `ANN.Test()` | Separate evaluation | Same concept |
| `ANN.Predict()` | `model.forward()` | Standard PyTorch |
| `ANN.PlotHistory()` | Custom Plotly plots | More customizable |
| `ANN.PlotConfusionMatrix()` | Custom Plotly plots | More customizable |
| `ANN.SaveModel()` | `torch.save()` | Standard PyTorch |
| `ANN.LoadModel()` | `torch.load()` | Standard PyTorch |

### Benefits of This Approach

- Full control over model architecture
- Direct access to PyTorch ecosystem
- Easy to extend with custom layers, losses, etc.
- Integration with PyTorch Lightning, Weights & Biases, etc.

### Applications

- Building type classification
- Geometric shape recognition
- Anomaly detection in architectural designs
- Property prediction from topology

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")